# AyurVision AI — Step 1: Dataset Download, Merge & Preprocessing

**What this notebook does:**
1. Mounts Google Drive (so nothing is lost when Colab resets)
2. Installs required libraries
3. Downloads 3 medicinal plant datasets from Kaggle
4. Explores each dataset (class names, image counts, sample images)
5. Merges all datasets into one unified folder structure
6. Resolves name conflicts across datasets
7. Preprocesses images (resize, normalize, quality filter)
8. Applies augmentation to build up small classes
9. Splits into train / val / test sets
10. Saves final dataset summary

**Runtime:** GPU T4 (free Colab tier) — estimated 45-60 minutes

---

## Cell 1 — Mount Google Drive
All output is saved to Drive so it survives Colab session resets.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os

# Base folder for the entire project on your Drive
BASE_DIR = '/content/drive/MyDrive/AyurVision'

# Sub-folders
RAW_DIR      = os.path.join(BASE_DIR, 'data/raw')        # raw downloaded datasets
MERGED_DIR   = os.path.join(BASE_DIR, 'data/merged')     # merged before preprocessing
FINAL_DIR    = os.path.join(BASE_DIR, 'data/final')      # train/val/test splits
LOGS_DIR     = os.path.join(BASE_DIR, 'logs')            # CSV reports

for d in [RAW_DIR, MERGED_DIR, FINAL_DIR, LOGS_DIR]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted. Folders ready:')
for d in [RAW_DIR, MERGED_DIR, FINAL_DIR, LOGS_DIR]:
    print(' ', d)

## Cell 2 — Install Libraries

In [ ]:
!pip install -q kaggle albumentations opencv-python-headless Pillow tqdm scikit-learn matplotlib seaborn
print('All libraries installed.')

## Cell 3 — Setup Kaggle API

**Before running this cell:**
1. Go to kaggle.com → your profile → Settings → API → Create New Token
2. This downloads `kaggle.json` to your computer
3. Upload that file when prompted below

In [ ]:
from google.colab import files
import os, shutil

# Upload kaggle.json
print('Please upload your kaggle.json file:')
uploaded = files.upload()

# Place it where the Kaggle CLI expects it
os.makedirs('/root/.kaggle', exist_ok=True)
shutil.move('kaggle.json', '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

print('Kaggle API configured successfully.')

# Verify
!kaggle datasets list --search 'medicinal plant' --max-size 500 | head -5

## Cell 4 — Download Dataset 1: Indian Medicinal Leaves (80 classes)

In [ ]:
import os

DS1_DIR = os.path.join(RAW_DIR, 'dataset1_kaggle_80classes')
os.makedirs(DS1_DIR, exist_ok=True)

print('Downloading Dataset 1 — Indian Medicinal Leaves (80 classes)...')
!kaggle datasets download -d aryashah2k/indian-medicinal-leaves-dataset \
    --path {DS1_DIR} --unzip

# Check what was downloaded
print('\nContents of Dataset 1:')
for root, dirs, files_list in os.walk(DS1_DIR):
    level = root.replace(DS1_DIR, '').count(os.sep)
    if level < 2:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files_list)} files)')

print('\nDataset 1 download complete.')

## Cell 5 — Download Dataset 2: Indian Medicinal Plant Dataset (40 species, 5945 imgs)

In [ ]:
DS2_DIR = os.path.join(RAW_DIR, 'dataset2_kaggle_40species')
os.makedirs(DS2_DIR, exist_ok=True)

print('Downloading Dataset 2 — Indian Medicinal Plant Dataset (40 species)...')
!kaggle datasets download -d warcoder/indian-medicinal-plant-image-dataset \
    --path {DS2_DIR} --unzip

print('\nContents of Dataset 2:')
for root, dirs, files_list in os.walk(DS2_DIR):
    level = root.replace(DS2_DIR, '').count(os.sep)
    if level < 2:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files_list)} files)')

print('\nDataset 2 download complete.')

## Cell 6 — Download Dataset 3: Segmented Medicinal Leaf Images (YOLO-ready)

In [ ]:
DS3_DIR = os.path.join(RAW_DIR, 'dataset3_kaggle_segmented')
os.makedirs(DS3_DIR, exist_ok=True)

print('Downloading Dataset 3 — Segmented Medicinal Leaf Images...')
!kaggle datasets download -d riteshranjansaroj/segmented-medicinal-leaf-images \
    --path {DS3_DIR} --unzip

print('\nContents of Dataset 3:')
for root, dirs, files_list in os.walk(DS3_DIR):
    level = root.replace(DS3_DIR, '').count(os.sep)
    if level < 2:
        indent = ' ' * 2 * level
        print(f'{indent}{os.path.basename(root)}/ ({len(files_list)} files)')

print('\nDataset 3 download complete.')

## Cell 7 — Explore Each Dataset: Class Names, Image Counts, Sample Images

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from PIL import Image
import numpy as np

def explore_dataset(dataset_dir, dataset_name):
    """
    Walk a dataset folder, find all class folders,
    count images per class, show samples.
    Returns: dict of {class_name: [list_of_image_paths]}
    """
    IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
    class_data = {}

    # Find the level where class folders live
    for root, dirs, files_list in os.walk(dataset_dir):
        imgs = [f for f in files_list
                if os.path.splitext(f)[1].lower() in IMAGE_EXTENSIONS]
        if imgs:
            class_name = os.path.basename(root).strip()
            full_paths = [os.path.join(root, f) for f in imgs]
            if class_name not in class_data:
                class_data[class_name] = []
            class_data[class_name].extend(full_paths)

    total_images = sum(len(v) for v in class_data.values())
    print(f'\n{'='*55}')
    print(f'Dataset: {dataset_name}')
    print(f'  Classes found : {len(class_data)}')
    print(f'  Total images  : {total_images}')
    print(f'  Min per class : {min(len(v) for v in class_data.values())}')
    print(f'  Max per class : {max(len(v) for v in class_data.values())}')
    print(f'  Avg per class : {total_images // len(class_data)}')
    print(f'\n  Classes:')
    for cls, paths in sorted(class_data.items()):
        print(f'    {cls:<35} {len(paths):>5} images')

    # Show 6 sample images
    sample_classes = list(class_data.keys())[:6]
    fig, axes = plt.subplots(2, 3, figsize=(12, 7))
    fig.suptitle(f'{dataset_name} — sample images', fontsize=13)
    for ax, cls in zip(axes.flatten(), sample_classes):
        try:
            img = Image.open(class_data[cls][0]).convert('RGB')
            ax.imshow(img)
            ax.set_title(cls, fontsize=9)
            ax.axis('off')
        except Exception:
            ax.axis('off')
    plt.tight_layout()
    plt.show()

    return class_data

# Explore all three datasets
ds1_data = explore_dataset(DS1_DIR, 'Dataset 1 — 80 classes (Kaggle aryashah2k)')
ds2_data = explore_dataset(DS2_DIR, 'Dataset 2 — 40 species (Kaggle warcoder)')
ds3_data = explore_dataset(DS3_DIR, 'Dataset 3 — Segmented leaves (Kaggle riteshranjansaroj)')

## Cell 8 — Build Class Name Mapping
Different datasets use different folder names for the same plant.
This cell normalizes them into a single canonical name.

In [ ]:
# Canonical name mapping — add more as you discover conflicts
# Format: 'folder_name_as_found': 'canonical_name'
NAME_MAP = {
    # Tulsi variations
    'Tulsi':                   'Tulsi',
    'tulsi':                   'Tulsi',
    'Holy Basil':              'Tulsi',
    'Ocimum tenuiflorum':      'Tulsi',

    # Neem variations
    'Neem':                    'Neem',
    'neem':                    'Neem',
    'Azadirachta indica':      'Neem',

    # Amla variations
    'Amla':                    'Amla',
    'amla':                    'Amla',
    'Indian Gooseberry':       'Amla',
    'Phyllanthus emblica':     'Amla',

    # Ashwagandha
    'Ashwagandha':             'Ashwagandha',
    'ashwagandha':             'Ashwagandha',
    'Withania somnifera':      'Ashwagandha',

    # Brahmi
    'Brahmi':                  'Brahmi',
    'brahmi':                  'Brahmi',
    'Bacopa monnieri':         'Brahmi',

    # Aloe Vera
    'Aloe Vera':               'Aloe_Vera',
    'aloe vera':               'Aloe_Vera',
    'Aloe vera':               'Aloe_Vera',

    # Curry Leaf
    'Curry Leaf':              'Curry_Leaf',
    'curry leaf':              'Curry_Leaf',
    'Murraya koenigii':        'Curry_Leaf',

    # Add more mappings as you discover them from Cell 7 output
}

def normalize_name(raw_name):
    """
    Normalize a class name using the mapping.
    If not found, clean it up and return as-is.
    """
    if raw_name in NAME_MAP:
        return NAME_MAP[raw_name]
    # Clean up: strip spaces, replace spaces with underscore, title case
    cleaned = raw_name.strip().replace(' ', '_').title()
    return cleaned

# Preview normalization for all three datasets
all_raw_classes = set()
for ds in [ds1_data, ds2_data, ds3_data]:
    all_raw_classes.update(ds.keys())

print(f'Total unique raw class names across all datasets: {len(all_raw_classes)}')
print('\nNormalization preview (raw -> canonical):')
for raw in sorted(all_raw_classes):
    canonical = normalize_name(raw)
    changed = ' ← renamed' if canonical != raw else ''
    print(f'  {raw:<40} -> {canonical}{changed}')

## Cell 9 — Merge All Datasets Into One Folder
Each class gets one folder. Images from all datasets are combined.
Images are renamed to avoid filename conflicts.

In [ ]:
import shutil
from tqdm import tqdm

# Clear merged dir if re-running
if os.path.exists(MERGED_DIR):
    shutil.rmtree(MERGED_DIR)
os.makedirs(MERGED_DIR, exist_ok=True)

def merge_dataset(class_data, dataset_label, counters):
    """
    Copy all images from a dataset into MERGED_DIR,
    using canonical class names as folder names.
    counters: dict tracking how many images per class were already copied.
    """
    VALID_EXT = {'.jpg', '.jpeg', '.png', '.bmp', '.webp', '.tiff'}
    copied = 0
    skipped = 0

    for raw_class, img_paths in tqdm(class_data.items(),
                                      desc=f'Merging {dataset_label}'):
        canonical = normalize_name(raw_class)
        dest_class_dir = os.path.join(MERGED_DIR, canonical)
        os.makedirs(dest_class_dir, exist_ok=True)

        for src_path in img_paths:
            ext = os.path.splitext(src_path)[1].lower()
            if ext not in VALID_EXT:
                skipped += 1
                continue

            # Unique filename: dataset_label + counter + extension
            idx = counters.get(canonical, 0)
            new_filename = f'{dataset_label}_{idx:05d}{ext}'
            dest_path = os.path.join(dest_class_dir, new_filename)

            shutil.copy2(src_path, dest_path)
            counters[canonical] = idx + 1
            copied += 1

    print(f'  {dataset_label}: {copied} images copied, {skipped} skipped')
    return counters

counters = {}
counters = merge_dataset(ds1_data, 'ds1', counters)
counters = merge_dataset(ds2_data, 'ds2', counters)
counters = merge_dataset(ds3_data, 'ds3', counters)

# Summary
merged_classes = sorted(os.listdir(MERGED_DIR))
print(f'\nMerge complete.')
print(f'Total classes in merged dataset : {len(merged_classes)}')
print(f'Total images                    : {sum(counters.values())}')
print(f'\nPer-class counts:')
for cls in merged_classes:
    n = len(os.listdir(os.path.join(MERGED_DIR, cls)))
    bar = '█' * (n // 20)
    print(f'  {cls:<40} {n:>5}  {bar}')

## Cell 10 — Image Quality Filter
Remove images that are:
- Corrupt / unreadable
- Too small (below 64×64 pixels)
- Too blurry (Laplacian variance below threshold)
- Not RGB (grayscale, RGBA etc.) — converted or skipped

In [ ]:
import cv2
from PIL import Image
import os
from tqdm import tqdm

MIN_SIZE    = 64       # minimum width and height in pixels
BLUR_THRESH = 50.0     # Laplacian variance below this = blurry

removed_corrupt  = 0
removed_small    = 0
removed_blurry   = 0
converted_rgba   = 0
kept             = 0
bad_files        = []

all_img_paths = []
for cls in os.listdir(MERGED_DIR):
    cls_dir = os.path.join(MERGED_DIR, cls)
    if os.path.isdir(cls_dir):
        for f in os.listdir(cls_dir):
            all_img_paths.append(os.path.join(cls_dir, f))

print(f'Checking {len(all_img_paths)} images for quality...')

for img_path in tqdm(all_img_paths, desc='Quality filter'):
    try:
        # Try opening with PIL
        pil_img = Image.open(img_path)

        # Convert RGBA or palette to RGB
        if pil_img.mode != 'RGB':
            pil_img = pil_img.convert('RGB')
            pil_img.save(img_path)  # overwrite with RGB version
            converted_rgba += 1

        w, h = pil_img.size

        # Size check
        if w < MIN_SIZE or h < MIN_SIZE:
            os.remove(img_path)
            removed_small += 1
            bad_files.append((img_path, 'too_small', f'{w}x{h}'))
            continue

        # Blur check using OpenCV Laplacian variance
        cv_img = cv2.imread(img_path)
        if cv_img is None:
            os.remove(img_path)
            removed_corrupt += 1
            bad_files.append((img_path, 'corrupt', ''))
            continue

        gray   = cv2.cvtColor(cv_img, cv2.COLOR_BGR2GRAY)
        lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()

        if lap_var < BLUR_THRESH:
            os.remove(img_path)
            removed_blurry += 1
            bad_files.append((img_path, 'blurry', f'{lap_var:.1f}'))
            continue

        kept += 1

    except Exception as e:
        os.remove(img_path)
        removed_corrupt += 1
        bad_files.append((img_path, 'exception', str(e)))

print(f'\nQuality filter complete:')
print(f'  Kept            : {kept}')
print(f'  Removed corrupt : {removed_corrupt}')
print(f'  Removed too small: {removed_small}')
print(f'  Removed blurry  : {removed_blurry}')
print(f'  Converted RGBA  : {converted_rgba}')
print(f'  Total removed   : {removed_corrupt + removed_small + removed_blurry}')

## Cell 11 — Augmentation for Small Classes
Any class with fewer than MIN_IMAGES images gets augmented
until it reaches MIN_IMAGES. This balances the dataset.

In [ ]:
import albumentations as A
import cv2
import numpy as np
from tqdm import tqdm
import os

MIN_IMAGES = 300   # minimum images per class after augmentation

# Augmentation pipeline — realistic real-world variation
aug_pipeline = A.Compose([
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomRotate90(p=0.5),
    A.ShiftScaleRotate(shift_limit=0.1,
                       scale_limit=0.15,
                       rotate_limit=30,
                       p=0.7),
    A.RandomBrightnessContrast(brightness_limit=0.3,
                                contrast_limit=0.3, p=0.6),
    A.HueSaturationValue(hue_shift_limit=15,
                          sat_shift_limit=25,
                          val_shift_limit=20, p=0.5),
    A.GaussNoise(var_limit=(10, 50), p=0.4),        # simulates dust/grain
    A.MotionBlur(blur_limit=5, p=0.2),              # simulates camera shake
    A.CoarseDropout(max_holes=8,
                    max_height=16,
                    max_width=16, p=0.3),            # simulates partial occlusion
    A.CLAHE(p=0.3),                                  # enhance local contrast
    A.RandomShadow(p=0.2),                           # simulates outdoor shadows
])

total_augmented = 0

for cls in tqdm(sorted(os.listdir(MERGED_DIR)), desc='Augmenting classes'):
    cls_dir = os.path.join(MERGED_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue

    existing = [f for f in os.listdir(cls_dir)
                if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
    current_count = len(existing)

    if current_count >= MIN_IMAGES:
        print(f'  {cls}: {current_count} images — OK, no augmentation needed')
        continue

    needed = MIN_IMAGES - current_count
    aug_idx = 0

    while aug_idx < needed:
        # Pick a random source image
        src_filename = existing[aug_idx % current_count]
        src_path = os.path.join(cls_dir, src_filename)

        img = cv2.imread(src_path)
        if img is None:
            aug_idx += 1
            continue

        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        augmented = aug_pipeline(image=img_rgb)['image']
        aug_bgr = cv2.cvtColor(augmented, cv2.COLOR_RGB2BGR)

        save_path = os.path.join(cls_dir, f'aug_{aug_idx:05d}.jpg')
        cv2.imwrite(save_path, aug_bgr, [cv2.IMWRITE_JPEG_QUALITY, 92])

        aug_idx += 1
        total_augmented += 1

    final_count = len(os.listdir(cls_dir))
    print(f'  {cls}: {current_count} -> {final_count} images (+{needed} augmented)')

print(f'\nTotal augmented images generated: {total_augmented}')

## Cell 12 — Resize All Images to 224×224
Standardize all images to the input size expected by SwinV2-B and EfficientNet.
Uses LANCZOS resampling for best quality.

In [ ]:
from PIL import Image
from tqdm import tqdm
import os

TARGET_SIZE = (224, 224)
resized_count = 0
error_count   = 0

all_paths = []
for cls in os.listdir(MERGED_DIR):
    cls_dir = os.path.join(MERGED_DIR, cls)
    if os.path.isdir(cls_dir):
        for f in os.listdir(cls_dir):
            if f.lower().endswith(('.jpg', '.jpeg', '.png')):
                all_paths.append(os.path.join(cls_dir, f))

print(f'Resizing {len(all_paths)} images to {TARGET_SIZE}...')

for img_path in tqdm(all_paths, desc='Resizing'):
    try:
        img = Image.open(img_path).convert('RGB')
        img = img.resize(TARGET_SIZE, Image.LANCZOS)
        # Save as JPEG to standardize format and reduce size
        jpg_path = os.path.splitext(img_path)[0] + '.jpg'
        img.save(jpg_path, 'JPEG', quality=92)
        # Remove original if different extension
        if img_path != jpg_path and os.path.exists(img_path):
            os.remove(img_path)
        resized_count += 1
    except Exception as e:
        error_count += 1

print(f'Resized: {resized_count} | Errors: {error_count}')

## Cell 13 — Train / Validation / Test Split
Split each class into 80% train, 10% val, 10% test.
Stratified — same ratio for every class.

In [ ]:
import os
import shutil
import random
from tqdm import tqdm

TRAIN_RATIO = 0.80
VAL_RATIO   = 0.10
TEST_RATIO  = 0.10
SEED        = 42
random.seed(SEED)

# Clear final dir if re-running
for split in ['train', 'val', 'test']:
    split_dir = os.path.join(FINAL_DIR, split)
    if os.path.exists(split_dir):
        shutil.rmtree(split_dir)
    os.makedirs(split_dir)

split_summary = []

for cls in tqdm(sorted(os.listdir(MERGED_DIR)), desc='Splitting'):
    cls_dir = os.path.join(MERGED_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue

    all_imgs = [f for f in os.listdir(cls_dir)
                if f.lower().endswith('.jpg')]
    random.shuffle(all_imgs)

    n = len(all_imgs)
    n_train = int(n * TRAIN_RATIO)
    n_val   = int(n * VAL_RATIO)
    # test gets the remainder

    splits = {
        'train': all_imgs[:n_train],
        'val':   all_imgs[n_train:n_train + n_val],
        'test':  all_imgs[n_train + n_val:]
    }

    for split_name, files_list in splits.items():
        dest = os.path.join(FINAL_DIR, split_name, cls)
        os.makedirs(dest, exist_ok=True)
        for f in files_list:
            shutil.copy2(os.path.join(cls_dir, f),
                         os.path.join(dest, f))

    split_summary.append({
        'class':   cls,
        'total':   n,
        'train':   len(splits['train']),
        'val':     len(splits['val']),
        'test':    len(splits['test'])
    })

# Print summary
print(f'\nTrain/Val/Test split complete.')
print(f'{"Class":<40} {"Total":>7} {"Train":>7} {"Val":>6} {"Test":>6}')
print('-' * 70)
for row in split_summary:
    print(f'{row["class"]:<40} {row["total"]:>7} {row["train"]:>7} '
          f'{row["val"]:>6} {row["test"]:>6}')

totals = {
    'total': sum(r['total'] for r in split_summary),
    'train': sum(r['train'] for r in split_summary),
    'val':   sum(r['val']   for r in split_summary),
    'test':  sum(r['test']  for r in split_summary)
}
print('-' * 70)
print(f'{"TOTAL":<40} {totals["total"]:>7} {totals["train"]:>7} '
      f'{totals["val"]:>6} {totals["test"]:>6}')

## Cell 14 — Save Class Label Map (class name → integer index)

In [ ]:
import json

train_dir = os.path.join(FINAL_DIR, 'train')
classes = sorted(os.listdir(train_dir))

class_to_idx = {cls: idx for idx, cls in enumerate(classes)}
idx_to_class = {idx: cls for cls, idx in class_to_idx.items()}

label_map = {
    'class_to_idx': class_to_idx,
    'idx_to_class': idx_to_class,
    'num_classes': len(classes)
}

label_map_path = os.path.join(BASE_DIR, 'label_map.json')
with open(label_map_path, 'w') as f:
    json.dump(label_map, f, indent=2)

print(f'Label map saved: {label_map_path}')
print(f'Total classes  : {len(classes)}')
print('\nClass index mapping:')
for cls, idx in class_to_idx.items():
    print(f'  {idx:>4}  {cls}')

## Cell 15 — Dataset Statistics Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import os

train_dir = os.path.join(FINAL_DIR, 'train')
classes   = sorted(os.listdir(train_dir))
counts    = [len(os.listdir(os.path.join(train_dir, c))) for c in classes]

# Plot image counts per class
fig, axes = plt.subplots(1, 2, figsize=(18, max(6, len(classes) * 0.3)))

# Horizontal bar chart
ax = axes[0]
colors = ['#1D9E75' if c >= 200 else '#EF9F27' if c >= 100 else '#E24B4A'
          for c in counts]
bars = ax.barh(classes, counts, color=colors, edgecolor='none', height=0.7)
ax.set_xlabel('Number of images (train set)')
ax.set_title('Images per class after preprocessing + augmentation')
ax.axvline(x=200, color='green', linestyle='--', alpha=0.5, label='200 target')
ax.legend()
for bar, count in zip(bars, counts):
    ax.text(bar.get_width() + 2, bar.get_y() + bar.get_height()/2,
            str(count), va='center', fontsize=7)

# Distribution histogram
ax2 = axes[1]
ax2.hist(counts, bins=20, color='#534AB7', edgecolor='white', alpha=0.8)
ax2.set_xlabel('Images per class')
ax2.set_ylabel('Number of classes')
ax2.set_title('Distribution of class sizes')
ax2.axvline(np.mean(counts), color='orange', linestyle='--',
            label=f'Mean: {np.mean(counts):.0f}')
ax2.legend()

plt.tight_layout()
plot_path = os.path.join(LOGS_DIR, 'dataset_distribution.png')
plt.savefig(plot_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Plot saved: {plot_path}')

# Print stats
print(f'\nDataset statistics (train set):')
print(f'  Total classes    : {len(classes)}')
print(f'  Total images     : {sum(counts)}')
print(f'  Mean per class   : {np.mean(counts):.0f}')
print(f'  Min per class    : {min(counts)}')
print(f'  Max per class    : {max(counts)}')
print(f'  Classes < 100    : {sum(1 for c in counts if c < 100)}')
print(f'  Classes 100-299  : {sum(1 for c in counts if 100 <= c < 300)}')
print(f'  Classes ≥ 300    : {sum(1 for c in counts if c >= 300)}')

## Cell 16 — Save Final Dataset Summary CSV

In [ ]:
import pandas as pd
import json
import os

rows = []
for cls in sorted(os.listdir(os.path.join(FINAL_DIR, 'train'))):
    row = {'class_name': cls}
    for split in ['train', 'val', 'test']:
        split_cls_dir = os.path.join(FINAL_DIR, split, cls)
        if os.path.exists(split_cls_dir):
            row[split] = len(os.listdir(split_cls_dir))
        else:
            row[split] = 0
    row['total'] = row['train'] + row['val'] + row['test']
    rows.append(row)

df = pd.DataFrame(rows)
df['class_idx'] = range(len(df))
df = df[['class_idx', 'class_name', 'total', 'train', 'val', 'test']]

csv_path = os.path.join(LOGS_DIR, 'dataset_summary.csv')
df.to_csv(csv_path, index=False)

print('Dataset summary saved to:', csv_path)
print(f'\nFull summary:')
print(df.to_string(index=False))
print(f'\nTotals:')
print(f'  Train : {df.train.sum()}')
print(f'  Val   : {df.val.sum()}')
print(f'  Test  : {df.test.sum()}')
print(f'  TOTAL : {df.total.sum()}')
print(f'\nAll done! Your dataset is ready at:')
print(f'  {FINAL_DIR}')
print(f'  ├── train/  ({df.train.sum()} images)')
print(f'  ├── val/    ({df.val.sum()} images)')
print(f'  └── test/   ({df.test.sum()} images)')
print(f'\nNext step: Run the model training notebook (Step 2)')